# 04 — XGBoost Baseline

**Purpose**: Train an XGBoost model on tabular features from visits 1 and 2 and evaluate it on the test set.

All logic lives in `src/ml_baseline.py`. This notebook calls those functions and inspects the outputs.

**Role in the comparison**: This is the honest benchmark. Both the LLM and the joint embedding model need to be meaningfully compared against this. XGBoost sees the same visit 1 + visit 2 features plus delta features — it just processes them as a flat table rather than as a text description or as trajectory embeddings.

In [ ]:
import sys
from pathlib import Path

project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd

from src.ml_baseline import (
    encode_categorical_features,
    extract_features_and_labels,
    train_xgboost_classifier,
    evaluate_predictions,
    summarise_metrics,
    print_feature_importance,
    save_xgboost_predictions,
    FEATURE_COLUMNS,
)

RESULTS_DIR = project_root / "results"
print(f"Project root : {project_root}")

## 1. Load train and test sets

In [ ]:
train_df = pd.read_csv(project_root / "data" / "processed" / "train_participants.csv")
test_df  = pd.read_csv(project_root / "data" / "processed" / "test_participants.csv")
print(f"Train: {len(train_df)} participants")
print(f"Test : {len(test_df)} participants")

## 2. Encode categorical features and extract X, y

In [ ]:
train_encoded, test_encoded = encode_categorical_features(train_df, test_df)
X_train, y_train = extract_features_and_labels(train_encoded)
X_test,  y_test  = extract_features_and_labels(test_encoded)
print(f"X_train shape : {X_train.shape}")
print(f"X_test shape  : {X_test.shape}")
print(f"Positive labels in train : {y_train.sum()}")
print(f"Positive labels in test  : {y_test.sum()}")

## 3. Train XGBoost

In [ ]:
xgb_model = train_xgboost_classifier(X_train, y_train)

## 4. Evaluate on test set

In [ ]:
y_pred       = xgb_model.predict(X_test)
y_pred_proba = xgb_model.predict_proba(X_test)[:, 1]

metrics = evaluate_predictions(y_test, y_pred, y_pred_proba)
summarise_metrics(metrics, model_name="XGBoost")

## 5. Feature importance

Expect delta features (mmse_delta_v1_v2, nwbv_delta_v1_v2) to rank highly — rate of change is more predictive than any single value.

In [ ]:
feature_cols = FEATURE_COLUMNS + ["sex", "handedness"]
print_feature_importance(xgb_model, feature_cols)

## 6. Prediction vs actual per participant

In [ ]:
results_df = test_df[["subject_id", "participant_group", "cdr_worsened_after_v2"]].copy()
results_df["xgb_prediction"]      = y_pred
results_df["xgb_prediction_proba"] = y_pred_proba.round(3)
results_df

## 7. Save results

In [ ]:
save_xgboost_predictions(test_df, y_pred, y_pred_proba, metrics, RESULTS_DIR)